In [1]:
import pandas as pd
import jax as jax
import jax.numpy as jnp
from jax import grad

jax.config.update("jax_enable_x64", True)

In [2]:
insurance_df = pd.read_csv('insurance.csv')

In [3]:
insurance_df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [4]:
x_data = insurance_df.drop(columns=['charges'])
y_data = insurance_df['charges'].values

In [5]:
def linear_regression(params, x):
    return jnp.dot(x, params)

def least_squares_loss(params, x, y):
    predictions = linear_regression(params, x)

    return ((y - predictions) ** 2)

In [6]:
def prepare_data(x_df):
    x_processed = x_df.copy()
    x_processed['sex'] = x_processed['sex'].map({'male': 0, 'female': 1})
    x_processed['smoker'] = x_processed['smoker'].map({'no': 0, 'yes': 1})
    x_processed['region'] = x_processed['region'].map({'northwest': 0, 'northeast': 1, 'southwest': 2, 'southeast': 3})

    return x_processed

In [7]:
key = jax.random.PRNGKey(42)
params = jax.random.normal(key, (x_data.shape[1],))
x_data = prepare_data(x_data)
x_jax = jnp.array(x_data.values)
y_jax = jnp.array(y_data)


In [8]:
print("Input features shape:", x_jax.shape)
print("Parameters shape:", params.shape)
print("Target values shape:", y_jax.shape)

Input features shape: (1338, 6)
Parameters shape: (6,)
Target values shape: (1338,)


In [9]:
res = linear_regression(params, x_data.values)
print("Initial predictions:", res)

Initial predictions: [ 0.76378773  4.70728044  3.94075333 ...  2.50095202 -0.48782537
 -7.51339958]


In [10]:
# LSE
initial_loss = least_squares_loss(params=params,x=x_jax,y=y_jax)

In [11]:
import jax.numpy as jnp

def normal_equation_inv(X, y):
    term1 = jnp.linalg.inv(jnp.dot(X.T, X))
    term2 = jnp.dot(X.T, y)
    return jnp.dot(term1, term2)

In [12]:
# Minimize LSE using normal equation
params = normal_equation_inv(x_jax, y_jax)
print("Parameters from normal equation:", params)
print("Predictions from normal equation:", linear_regression(params, x_jax))


Parameters from normal equation: [  199.97941995  -755.06673478    53.773517     238.04769336
 23308.8030125   -336.53454006]
Predictions from normal equation: [27180.55730088  4644.00530128  7078.48927942 ...  3816.51330549
  4158.78874259 36315.67703382]


In [13]:
N = x_jax.shape[0]  
# RSS (Residual Sum of Squares) 
rss = jnp.sum(least_squares_loss(params, x_jax, y_jax))

# 1. Mean Squared Error (MSE)
mse = rss / N

# 2. Root Mean Squared Error (RMSE)
rmse = jnp.sqrt(mse)

# R-squared (R2)
y_mean = jnp.mean(y_jax)
total_variance = jnp.sum((y_jax - y_mean) ** 2)
r2_score = 1 - (rss / total_variance)

print(f"RSS (Residual Sum of Squares): {rss:,.0f}")
print(f"Mean Squared Error (MSE):      {mse:,.0f}")
print(f"RMSE (Average Error):          ${rmse:,.2f}")
print(f"R^2 Score (Accuracy):          {r2_score:.4f}")

RSS (Residual Sum of Squares): 54,776,985,993
Mean Squared Error (MSE):      40,939,451
RMSE (Average Error):          $6,398.39
R^2 Score (Accuracy):          0.7206
